# Dataset Loading

Pull data from huggingface and download

In [1]:
import tarfile
import json
import pandas as pd

In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="yann111/GlobalGeoTree",
    repo_type="dataset",
    local_dir="./GlobalGeoTree-6M",
    allow_patterns="GlobalGeoTree-6M/*"  # Only files in this folder
)

print("Download complete!")


c:\Users\yy21473\.GitHubFiles\dst-group-work\GlobalGeoTree\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 118 files: 100%|██████████| 118/118 [01:52<00:00,  1.05it/s]

Download complete!


# Parse dataset to dataframe

## For single tar

In [ ]:

tar_path = "GlobalGeoTree-6M/GlobalGeoTree-6M/GGT-0.2M_0.5M-000000.tar"

small_sample = []

with tarfile.open(tar_path, "r") as tar:
    for member in tar:
        if member.isfile() and member.name.endswith(".json"):
            f = tar.extractfile(member)
            if f:
                obj = json.loads(f.read().decode("utf-8"))
                small_sample.append(obj)

                if len(small_sample) >= 1000:
                    break

# Merge every two consecutive JSON objects
merged_records = []
for i in range(0, len(small_sample), 2):
    if i+1 < len(small_sample):
        merged = {**small_sample[i], **small_sample[i+1]}
        merged_records.append(merged)
    else:
        merged_records.append(small_sample[i])  # If odd, keep last as is

df = pd.DataFrame(merged_records)


## For all tars

In [2]:
from pathlib import Path
from tqdm import tqdm

def process_tar(path):
    small_sample = []
    with tarfile.open(path, "r") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(".json"):
                f = tar.extractfile(member)
                if f:
                    obj = json.loads(f.read().decode("utf-8"))
                    small_sample.append(obj)

        # Merge every two consecutive JSON objects
        merged_records = []
        for i in range(0, len(small_sample), 2):
            if i+1 < len(small_sample):
                merged = {**small_sample[i], **small_sample[i+1]}
                merged_records.append(merged)
            else:
                merged_records.append(small_sample[i])  # If odd, keep last as is

        df = pd.DataFrame(merged_records)
        return df

folder = Path("GlobalGeoTree-6M/GlobalGeoTree-6M/")

dfAll = pd.DataFrame()

tar_files = list(folder.glob("*.tar"))  # so tqdm knows the total
dfAll = pd.DataFrame()

for tar_file in tqdm(tar_files, desc="Processing tar files"):
    dfTemp = process_tar(tar_file)
    dfAll = pd.concat([dfAll, dfTemp], ignore_index=True)

Processing tar files: 100%|██████████| 118/118 [18:07<00:00,  9.21s/it]


# Save to Parquet

In [3]:
dfAll.to_parquet("GlobalGeoTree-6M/GlobalGeoTree-6M.parquet", index=False)